# 对LLM基本代码使用用法的小结

## 基本环境的准备

In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, PreTrainedModel, modeling_outputs
import torch
from loguru import logger
import os
import sys
from typing import cast

os.environ["HTTP_PROXY"] = "http://127.0.0.1:6382"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:6382"

logger.remove()
logger.add(sys.stdout, level="INFO", colorize=True)

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 基础总结1：使用 GPT-2 完成下一个 Token 预测与句子生成

In [2]:
class GPT2GenerateToken:
    def __init__(self, tokenizer: GPT2Tokenizer, model: PreTrainedModel) -> None:
        self._tokenizer: GPT2Tokenizer = tokenizer
        self._model: PreTrainedModel = model
        self._top_k: int = 3  # 默认topk设置为3

    def generate_next_token(self, prompt: str) -> None:
        """
        模型生成基本的下一个token
        :param prompt: 提示词字符串
        :return: None
        """
        logger.info(f"Step1: 从提示词字符串->对应的token id列表->对应的张量类形式->分词后的tokens列表")
        logger.info(f"prompt : {prompt}")
        input_ids: list[int] = self._tokenizer.encode(prompt)
        logger.info(f"对应的token ids ：{input_ids}")
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)
        logger.info(f"对应的张量类的表示：{token_ids}")
        tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids]
        logger.info(f"对应的分开的词的形式：{tokens}")

        with torch.no_grad():
            output: modeling_outputs.CausalLMOutputWithCrossAttentions = self._model(token_ids)
            logits: torch.Tensor = cast(torch.Tensor, output.logits)
            logger.info(
                f"logits张量的shape为：{logits.shape}，其中的第一个维度是batch，第二个维度是token数，第三个维度是词表数")
            next_token_logits: torch.Tensor = logits[0, -1, :]
            logger.info(f"用于打分的logits维度是{next_token_logits.shape}\n值为{next_token_logits}")
            next_token_probabilities: torch.Tensor = torch.softmax(next_token_logits, dim=0)
            logger.info(f"经过softmax后的概率shape：{next_token_probabilities.shape}，概率是{next_token_probabilities}")
            top_k_probabilities, top_k_index = torch.topk(next_token_probabilities, self._top_k)
            top_k_tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in top_k_index]
            for i, (probability, index, token) in enumerate(zip(top_k_probabilities, top_k_index, top_k_tokens)):
                logger.info(f"Top {i} : probability = {probability}, index = {index}, token = {token}")

    def generate_sentence(self, prompt: str, max_token_num: int = 50) -> str:
        """
        模型进行基本的续写
        :param max_token_num: 最大句子长度限制
        :param prompt: 提示词字符串
        :return: 续写的字符串
        """
        # process basic input
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)

        # logits and probabilities, use non-greedy settings
        should_generate_end: bool = False
        while not should_generate_end:
            with torch.no_grad():
                logits: torch.Tensor = cast(torch.Tensor, self._model(token_ids).logits)[0, -1, :]
                probabilities: torch.Tensor = torch.softmax(logits, dim=0)
                next_token_id: int = cast(int, torch.argmax(probabilities).item())
                next_token: str = cast(str, self._tokenizer.decode(next_token_id))

                prompt += next_token
                token_ids = torch.cat([token_ids, torch.tensor([[next_token_id]], dtype=torch.long)], dim=1)

                if next_token in [".", "?", "!"] or token_ids.shape[1] >= max_token_num:
                    should_generate_end = True
        return prompt

展示GPT-2生成下一个token的详细过程

In [3]:
gpt2_generate_token = GPT2GenerateToken(gpt2_tokenizer, gpt2_model)
gpt2_generate_token.generate_next_token("Thank you very")

2026-08-17 13:09:22.978 | INFO     | __main__:generate_next_token:13 - Step1: 从提示词字符串->对应的token id列表->对应的张量类形式->分词后的tokens列表
2026-08-17 13:09:22.978 | INFO     | __main__:generate_next_token:14 - prompt : Thank you very
2026-08-17 13:09:22.979 | INFO     | __main__:generate_next_token:16 - 对应的token ids ：[10449, 345, 845]
2026-08-17 13:09:22.980 | INFO     | __main__:generate_next_token:18 - 对应的张量类的表示：tensor([[10449,   345,   845]])
2026-08-17 13:09:22.981 | INFO     | __main__:generate_next_token:20 - 对应的分开的词的形式：['Thank', ' you', ' very']
2026-08-17 13:09:23.019 | INFO     | __main__:generate_next_token:25 - logits张量的shape为：torch.Size([1, 3, 50257])，其中的第一个维度是batch，第二个维度是token数，第三个维度是词表数
2026-08-17 13:09:23.019 | INFO     | __main__:generate_next_token:28 - 用于打分的logits维度是torch.Size([50257])
值为tensor([-59.0983, -58.9518, -67.3273,  ..., -68.9031, -69.3630, -63.3151])
2026-08-17 13:09:23.020 | INFO     | __main__:generate_next_token:30 - 经过softmax后的概率shape：torch.Size([50257])，概率是tensor([1

使用GPT-2进行句子续写，使用greedy settings

In [4]:
answer: str = gpt2_generate_token.generate_sentence("The meaning of life is", max_token_num=50)
logger.info(answer)

2026-08-17 13:09:23.150 | INFO     | __main__:<module>:2 - The meaning of life is not the same as the meaning of death.


## 基础总结2：GPT-2 对下一个 Token 的打分与采样过程

增加温度以及non-greedy的考虑

In [5]:
class GPT2GenerateTokenWithTemperature(GPT2GenerateToken):
    """
    考虑温度的打分过程
    """

    def __init__(self, tokenizer: GPT2Tokenizer, model: PreTrainedModel, temperature: float) -> None:
        super().__init__(tokenizer, model)
        self._temperature = temperature

    @property
    def temperature(self) -> float:
        return self._temperature

    @temperature.setter
    def temperature(self, value: float) -> None:
        if value <= 0:
            raise ValueError("temperature must be greater than 0")
        self._temperature = value

    def generate_next_token(self, prompt: str, is_greedy: bool = False) -> str:
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)

        with torch.no_grad():
            logits: torch.Tensor = cast(torch.Tensor, self._model(token_ids).logits)[0, -1, :]
            scaled_logits: torch.Tensor = logits / self._temperature
            token_probabilities: torch.Tensor = torch.softmax(scaled_logits, dim=0)

            if is_greedy:
                next_token, _ = self._get_greedy_next_token_with_id(token_probabilities)
            else:
                next_token, _ = self._get_non_greedy_next_token_with_id(token_probabilities)
            return next_token

    def generate_sentence(self, prompt: str, is_greedy: bool = False, max_token_num: int = 50) -> str:
        """
        模型进行基本的续写
        :param is_greedy: 是否采用greedy settings
        :param max_token_num: 最大句子长度限制
        :param prompt: 提示词字符串
        :return: 续写的字符串
        """
        # process basic input
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)

        # logits and probabilities, use non-greedy settings
        should_generate_end: bool = False
        while not should_generate_end:
            with torch.no_grad():
                logits: torch.Tensor = cast(torch.Tensor, self._model(token_ids).logits)[0, -1, :]
                logits: torch.Tensor = logits / self._temperature
                token_probabilities: torch.Tensor = torch.softmax(logits, dim=0)
                if is_greedy:
                    next_token, next_token_id = self._get_greedy_next_token_with_id(token_probabilities)
                else:
                    next_token, next_token_id = self._get_non_greedy_next_token_with_id(token_probabilities)

                prompt += next_token
                token_ids = torch.cat([token_ids, torch.tensor([[next_token_id]], dtype=torch.long)], dim=1)

                if next_token in [".", "?", "!"] or token_ids.shape[1] >= max_token_num:
                    should_generate_end = True
        return prompt

    def _get_greedy_next_token_with_id(self, probabilities: torch.Tensor) -> tuple[str, int]:
        next_token_id: int = cast(int, torch.argmax(probabilities).item())
        return cast(str, self._tokenizer.decode(next_token_id)), next_token_id

    def _get_non_greedy_next_token_with_id(self, probabilities: torch.Tensor) -> tuple[str, int]:
        next_token_id: int = cast(int, torch.multinomial(probabilities, 1).item())
        return cast(str, self._tokenizer.decode(next_token_id)), next_token_id


对不同温度以及是否是non-greedy的情况进行展示

In [6]:
gpt2_generate_token_with_temperature = GPT2GenerateTokenWithTemperature(gpt2_tokenizer, gpt2_model, 1.0)
logger.info("test case 1 : temperature = 1.0, non-greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is")
logger.info(input_prompt)
logger.info('=' * 20)

logger.info("test case 2 : temperature = 1.0, greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is", is_greedy=True)
logger.info(input_prompt)
logger.info('=' * 20)

gpt2_generate_token_with_temperature.temperature = 0.3
logger.info("test case 3 : temperature = 0.3, non-greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is")
logger.info(input_prompt)
logger.info('=' * 20)

logger.info("test case 4 : temperature = 0.3, greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is", is_greedy=True)
logger.info(input_prompt)
logger.info('=' * 20)

gpt2_generate_token_with_temperature.temperature = 1.5
logger.info("test case 5 : temperature = 1.5, non-greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is")
logger.info(input_prompt)
logger.info('=' * 20)

logger.info("test case 6 : temperature = 1.5, greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is", is_greedy=True)
logger.info(input_prompt)
logger.info('=' * 20)

2026-08-17 13:09:23.168 | INFO     | __main__:<module>:2 - test case 1 : temperature = 1.0, non-greedy settings
2026-08-17 13:09:23.223 | INFO     | __main__:<module>:4 - The meaning of life is an existential commitment.
2026-08-17 13:09:23.223 | INFO     | __main__:<module>:5 - ====================
2026-08-17 13:09:23.223 | INFO     | __main__:<module>:7 - test case 2 : temperature = 1.0, greedy settings
2026-08-17 13:09:23.342 | INFO     | __main__:<module>:9 - The meaning of life is not the same as the meaning of death.
2026-08-17 13:09:23.343 | INFO     | __main__:<module>:10 - ====================
2026-08-17 13:09:23.343 | INFO     | __main__:<module>:13 - test case 3 : temperature = 0.3, non-greedy settings
2026-08-17 13:09:23.561 | INFO     | __main__:<module>:15 - The meaning of life is to live in a world where there is no one to blame but yourself.
2026-08-17 13:09:23.561 | INFO     | __main__:<module>:16 - ====================
2026-08-17 13:09:23.561 | INFO     | __main__:<mo

## 基础总结3：从文本到 Token、Embedding 与上下文 Feature

In [7]:
class GPT2TokenEmbeddingFeature:
    """
    用于展示GPT-2模型token对应的embedding以及feature的流程中的行为
    """

    def __init__(self, tokenizer: GPT2Tokenizer, model: GPT2LMHeadModel) -> None:
        self._tokenizer = tokenizer
        self._model = model
        self._wte_embedding: torch.Tensor = model.transformer.wte.weight.detach()
        self._wpe_embedding: torch.Tensor = model.transformer.wpe.weight.detach()

    def display_embedding_and_feature(self, prompt: str) -> None:
        """
        展示prompt在前向传播过程中形成embedding，feature的过程
        :param prompt:
        :return:
        """
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)
        tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids]

        with torch.no_grad():
            outputs = self._model(token_ids, output_hidden_states=True)
        hidden_states = outputs.hidden_states

        logger.info(f"input token nums = {len(tokens)}")
        for position, (token, token_id) in enumerate(zip(tokens, input_ids)):
            logger.info(f"{'=' * 20} token_{position} = {token!r} {'*' * 20}")
            wte: torch.Tensor = self._wte_embedding[token_id]
            wpe: torch.Tensor = self._wpe_embedding[position]
            logger.info(f"WTE[:3] = {wte[:3].tolist()}")
            logger.info(f"WPE[:3] = {wpe[:3].tolist()}")

            embedding: torch.Tensor = wte + wpe
            logger.info(f"embedding = {embedding[:3].tolist()}")

            hidden_state_zero: torch.Tensor = hidden_states[0][0, position, :]
            logger.info(f"hidden_state_zero = {hidden_state_zero[:3].tolist()}")
            logger.info(f" hidden_state_zero = embedding_vector !")
            hidden_state_low: torch.Tensor = hidden_states[1][0, position, :]
            hidden_state_mid: torch.Tensor = hidden_states[6][0, position, :]
            hidden_state_high: torch.Tensor = hidden_states[-1][0, position, :]
            logger.info(f"hidden_state_low[:3] = {hidden_state_low[:3].tolist()}")
            logger.info(f"hidden_state_mid[:3] = {hidden_state_mid[:3].tolist()}")
            logger.info(f"hidden_state_high[:3] = {hidden_state_high[:3].tolist()}")


下面的代码展示了从token到embedding再到各层feature的过程。其中embedding（wte+wpe）和feature的第0层是一致的。

In [8]:
gpt2_token_embedding_feature = GPT2TokenEmbeddingFeature(gpt2_tokenizer, cast(GPT2LMHeadModel, gpt2_model))
gpt2_token_embedding_feature.display_embedding_and_feature("I went to the bank")

2026-08-17 13:09:24.677 | INFO     | __main__:display_embedding_and_feature:26 - input token nums = 5
2026-08-17 13:09:24.677 | INFO     | __main__:display_embedding_and_feature:28 - ==================== token_0 = 'I' ********************
2026-08-17 13:09:24.677 | INFO     | __main__:display_embedding_and_feature:31 - WTE[:3] = [0.14739921689033508, -0.09585089236497879, 0.1429542899131775]
2026-08-17 13:09:24.677 | INFO     | __main__:display_embedding_and_feature:32 - WPE[:3] = [-0.01882071979343891, -0.19741860032081604, 0.004026724956929684]
2026-08-17 13:09:24.678 | INFO     | __main__:display_embedding_and_feature:35 - embedding = [0.12857849895954132, -0.29326948523521423, 0.14698101580142975]
2026-08-17 13:09:24.678 | INFO     | __main__:display_embedding_and_feature:38 - hidden_state_zero = [0.12857849895954132, -0.29326948523521423, 0.14698101580142975]
2026-08-17 13:09:24.678 | INFO     | __main__:display_embedding_and_feature:39 -  hidden_state_zero = embedding_vector !
202

## 基础总结4：WTE 与 WPE 如何共同构成 Transformer 的输入

本部分以同一个词在不同的位置、以及同一个位置的不同含义的词为例来展示transformer过程中对相同含义不同位置、相同位置不同含义的词的处理方式

In [9]:
class GPT2AnalogyAndWPE(GPT2TokenEmbeddingFeature):
    """
    用于展示GPT-2模型同一个token在不同位置以及同义词的区分情况
    """

    def __init__(self, tokenizer: GPT2Tokenizer, model: GPT2LMHeadModel) -> None:
        super().__init__(tokenizer, model)

    def display_wte_wpe_feature(self, prompt1: str, prompt2: str, target: str) -> None:
        """
        同一个 token、同一个语义，在不同position下，先看WPE如何改变输入embedding，再看这种差异经过Transformer后如何体现在feature上。
        :param prompt1: 提示词1
        :param prompt2: 提示词2
        :param target: 用于展示的目标词
        :return: None
        """
        input_ids1: list[int] = self._tokenizer.encode(prompt1)
        input_ids2: list[int] = self._tokenizer.encode(prompt2)
        token_ids1: torch.Tensor = torch.tensor([input_ids1], dtype=torch.long)
        token_ids2: torch.Tensor = torch.tensor([input_ids2], dtype=torch.long)
        tokens1: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids1]
        tokens2: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids2]

        pos1: int = next(i for i in range(len(tokens1)) if tokens1[i].strip().lower() == target.strip().lower())
        pos2: int = next(i for i in range(len(tokens2)) if tokens2[i].strip().lower() == target.strip().lower())
        wte1: torch.Tensor = self._wte_embedding[input_ids1[pos1]]
        wte2: torch.Tensor = self._wte_embedding[input_ids2[pos2]]
        wpe1: torch.Tensor = self._wpe_embedding[pos1]
        wpe2: torch.Tensor = self._wpe_embedding[pos2]
        embedding1: torch.Tensor = wte1 + wpe1
        embedding2: torch.Tensor = wte2 + wpe2

        logger.info(f"wte1[:3] = {wte1[:3].tolist()}")
        logger.info(f"wte2[:3] = {wte2[:3].tolist()}")
        logger.info(f"wpe1[:3] = {wpe1[:3].tolist()}")
        logger.info(f"wpe2[:3] = {wpe2[:3].tolist()}")
        logger.info(f"embedding1[:3] = {embedding1[:3].tolist()}")
        logger.info(f"embedding2[:3] = {embedding2[:3].tolist()}")

        with torch.no_grad():
            output1 = self._model(token_ids1, output_hidden_states=True).hidden_states
            output2 = self._model(token_ids2, output_hidden_states=True).hidden_states

        feature_low1 = output1[1][0, pos1, :]
        feature_low2 = output2[1][0, pos2, :]
        logger.info(f"feature_low1[:3] = {feature_low1[:3].tolist()}")
        logger.info(f"feature_low2[:3] = {feature_low2[:3].tolist()}")

        feature_mid1 = output1[5][0, pos1, :]
        feature_mid2 = output2[5][0, pos2, :]
        logger.info(f"feature_mid1[:3] = {feature_mid1[:3].tolist()}")
        logger.info(f"feature_mid2[:3] = {feature_mid2[:3].tolist()}")

        feature_high1 = output1[-1][0, pos1, :]
        feature_high2 = output2[-1][0, pos2, :]
        logger.info(f"feature_high1[:3] = {feature_high1[:3].tolist()}")
        logger.info(f"feature_high2[:3] = {feature_high2[:3].tolist()}")


首先展示同一个token在不同的位置时引起的差异

In [10]:
gpt2_analogy_and_wpe = GPT2AnalogyAndWPE(gpt2_tokenizer, cast(GPT2LMHeadModel, gpt2_model))
gpt2_analogy_and_wpe.display_wte_wpe_feature("I love my cat because they are cute", "Their cat is not happy today",
                                             "cat")

2026-08-17 13:09:24.709 | INFO     | __main__:display_wte_wpe_feature:33 - wte1[:3] = [0.0099662309512496, 0.03655250370502472, 0.16402631998062134]
2026-08-17 13:09:24.710 | INFO     | __main__:display_wte_wpe_feature:34 - wte2[:3] = [0.0099662309512496, 0.03655250370502472, 0.16402631998062134]
2026-08-17 13:09:24.710 | INFO     | __main__:display_wte_wpe_feature:35 - wpe1[:3] = [-0.0002833660691976547, -0.0738026350736618, 0.10552646964788437]
2026-08-17 13:09:24.710 | INFO     | __main__:display_wte_wpe_feature:36 - wpe2[:3] = [0.023959433659911156, -0.05379203334450722, -0.09487864375114441]
2026-08-17 13:09:24.710 | INFO     | __main__:display_wte_wpe_feature:37 - embedding1[:3] = [0.009682864882051945, -0.037250131368637085, 0.2695527970790863]
2026-08-17 13:09:24.710 | INFO     | __main__:display_wte_wpe_feature:38 - embedding2[:3] = [0.03392566367983818, -0.017239529639482498, 0.06914767622947693]
2026-08-17 13:09:24.741 | INFO     | __main__:display_wte_wpe_feature:46 - featu

再展示同一个词（甚至可以位于同一个位置），但含义不同时所展示出的差异

In [11]:
gpt2_analogy_and_wpe.display_wte_wpe_feature("I deposited money at the bank yesterday",
                                             "I saw water near the bank last week", "bank")

2026-08-17 13:09:24.763 | INFO     | __main__:display_wte_wpe_feature:33 - wte1[:3] = [0.045706383883953094, 0.0818503201007843, 0.13745230436325073]
2026-08-17 13:09:24.763 | INFO     | __main__:display_wte_wpe_feature:34 - wte2[:3] = [0.045706383883953094, 0.0818503201007843, 0.13745230436325073]
2026-08-17 13:09:24.763 | INFO     | __main__:display_wte_wpe_feature:35 - wpe1[:3] = [0.009602269157767296, -0.0338851697742939, 0.13123270869255066]
2026-08-17 13:09:24.764 | INFO     | __main__:display_wte_wpe_feature:36 - wpe2[:3] = [0.009602269157767296, -0.0338851697742939, 0.13123270869255066]
2026-08-17 13:09:24.764 | INFO     | __main__:display_wte_wpe_feature:37 - embedding1[:3] = [0.05530865490436554, 0.0479651503264904, 0.2686850130558014]
2026-08-17 13:09:24.764 | INFO     | __main__:display_wte_wpe_feature:38 - embedding2[:3] = [0.05530865490436554, 0.0479651503264904, 0.2686850130558014]
2026-08-17 13:09:24.793 | INFO     | __main__:display_wte_wpe_feature:46 - feature_low1[:3